Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle
import pyarrow as pa

Voy a guarda la tabla de las imputaciones en `res`

In [2]:
df_imputations_limpieza = pd.read_parquet("res/procesed/df_imputations_limpieza.parquet")

## Filtrar imputaciones por maquinas productivas
Todas las imputaciones que no pertenezcan a la selección de maquinas productivas (`grupo_maquinas`) debo hacerlas desaparecer ya que son imputaciones a no emplear para mi proyecto. 

In [3]:
# cargo el df de maquinas y grupos
maquina_grupo = pd.read_parquet("res//procesed/maquina_grupo.parquet")
display(maquina_grupo.sample(1))
print(f"Total de máquinas: {len(maquina_grupo)}")
print(f"Grupos de máquinas: {len(maquina_grupo["CodigoGrupo"].unique())}")

#print(len(maquina_grupo["CodigoMaquina"].unique()))
#print(len(maquina_grupo["CodigoMaquina"]))

,IDMaquina,CodigoMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
22,2cbba4c2-2da1-4b9d-8e39-b0e8cf0b5b6e,CM10,CV KONDIA B-1370,15ab8d51-9e37-4da9-a196-dd259f9ea224,CV-ALU,Centres de Mecanitzat Vertical ALUMINIS


Total de máquinas: 32
Grupos de máquinas: 9


> Para comprender mejor la siguiente celda del código, quiero remarcar que para `maquinas_validas[CodigoMaquina]` tiene su codigo homologo en `df_imputations_limpieza[smNodoId]`. Esto permite relacionar facilmente ambas tamblas mediante el código de la máquina. 

In [4]:
maquinas_validas = maquina_grupo["CodigoMaquina"].unique()
print(f"Imputaciones antes: {len(df_imputations_limpieza)}")
df_imput_productivas = df_imputations_limpieza[
    df_imputations_limpieza["smNodoId"].isin(maquinas_validas)
]
print(f"Imputaciones después: {len(df_imput_productivas)}")
print(f"Eliminadas (imputaciones en máquinas no productivas): {len(df_imputations_limpieza) - len(df_imput_productivas)}")

Imputaciones antes: 140485
Imputaciones después: 59574
Eliminadas (imputaciones en máquinas no productivas): 80911


In [5]:
fuera = round(((len(df_imputations_limpieza)-len(df_imput_productivas))/len(df_imputations_limpieza))*100, 2)
print(f"Desaparecen el {fuera}% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio")

Desaparecen el 57.59% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio


#### Revisar incidencias
Hay imputaciones en maquinas que corresponden a incidencias, inspecciono cuantas son y planteo descartarlas, ya que estas no tocan tiempo productivo.

In [6]:
# Revisar incidencias:
df_incidencias = df_imput_productivas[df_imput_productivas["smIncidenciaId"].notnull()]  
print(df_incidencias.shape)

(4, 15)


In [7]:
df_imput_productivas = df_imput_productivas[df_imput_productivas["smIncidenciaId"].isnull()]  

In [8]:
df_imputations_productivas = df_imput_productivas.shape[0]

## Resolver fases imputadas en N nodos
Interesa averiguar si se ha dado una misma fase en distintas maquinas, por ello debo verificar si todas las imputaciones de una misma fase se han hecho en la misma máquina.   
Para cada imputación podemos saber en que nodo se ha realizado, un nodo puede ser una maquina o un centro de trabajo.

In [9]:
maquinas_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()
print(f"Total de fases productivas: {len(df_imput_productivas.groupby('SmFaseUid'))}")
print("")
print("Entre las fases productivas, el número de máquinas distintas por fase es:")
print(maquinas_por_fase.value_counts())

Total de fases productivas: 19149

Entre las fases productivas, el número de máquinas distintas por fase es:
smNodoId
1    18250
2      860
3       37
4        2
Name: count, dtype: int64


In [10]:
# total fases con maquinas productivas
total_fases_productivas = maquinas_por_fase.shape[0]

Averiguar que maquinas son las que presentan imputaciones de una misma fase en distintos nodos es innecesario, lo que realmente me interesa es ver si este cambio de nodo traspasa entre distintos grupos de máquinas:

In [11]:
# recuento de nodos por fase (sobre imputaciones productivas)
nodos_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()

fases_mono = nodos_por_fase[nodos_por_fase == 1].index
fases_multi = nodos_por_fase[nodos_por_fase > 1].index

# --- 1. Peso en número de fases ---
total_fases = len(nodos_por_fase)
n_multi = len(fases_multi)
print(f"Fases totales:            {total_fases}")
print(f"Fases mono-nodo:          {len(fases_mono)} ({len(fases_mono)/total_fases:.1%})")
print(f"Fases multi-nodo:         {n_multi} ({n_multi/total_fases:.1%})")

# --- 2. Peso en tiempo real (¿son marginales también en volumen de trabajo?) ---
tiempo_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]["MinutosRealNbr"].sum()
tiempo_total = df_imput_productivas["MinutosRealNbr"].sum()
print(f"\nTiempo real en fases multi-nodo: {tiempo_multi/tiempo_total:.1%} del total")

Fases totales:            19149
Fases mono-nodo:          18250 (95.3%)
Fases multi-nodo:         899 (4.7%)

Tiempo real en fases multi-nodo: 15.6% del total


Hay pocas fases en multi-nodo (4,7%), pero con peso desproporcionado en el tiempo total de todas las imputaciones (15,6%). No son ruido marginal, sino fases más "pesadas" que la media.      
Entonces el 4,7% de las fases acumula el 15,6% del tiempo real. Eso significa que una fase multi-nodo tienen más tiempo real que una fase normal. Hablando con el equipo de producción tiene sentido lógico, son fases largas o complejas, y precisamente por eso se repartieron entre varias máquinas o sesiones.

> Voy a inspeccionarlas.

¿En que máquinas aparecen las fases multi nodo?

In [12]:
# ¿en qué máquinas aparecen las fases multi-nodo?
imput_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]

t = imput_multi.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
fraccion_dominante = t.groupby("SmFaseUid").max() / t.groupby("SmFaseUid").sum()
print(f"Total de fases donde la maquina dominante concentra >80% del tiempo: {(fraccion_dominante > 0.8).mean():.1%}")

Total de fases donde la maquina dominante concentra >80% del tiempo: 33.5%


En el 33,5% de las fases multi-nodo la máquina dominante concentra más del 80% del tiempo, es decir, en el 66,5% de las imputaciones en `df_imput_productivas`, el tiempo está genuinamente repartido, la "maquina dominante" no domina tanto, convive con otras máquinas con peso real.  
  
> Voy a trabajar con ese 33,5% de fases donde es facilmente identificable la maquina dominante, para las 66,5% restantes voy a prescindir, obviando las mismas para este proyecto. 

In [13]:
# tiempo por (fase, nodo)
tiempo_fase_nodo = df_imput_productivas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()

# fracción de la dominante en cada fase
dominante = tiempo_fase_nodo.groupby("SmFaseUid").max()
total = tiempo_fase_nodo.groupby("SmFaseUid").sum()
fraccion_dominante = dominante / total

# fases válidas: la dominante concentra >80% (esto incluye automáticamente las mono-nodo, que son 100%)
fases_validas = fraccion_dominante[fraccion_dominante > 0.8].index

print(f"Fases totales antes:     {fraccion_dominante.shape[0]}")
print(f"Fases válidas (dominante >80%): {len(fases_validas)}")
print(f"Fases descartadas (repartidas): {fraccion_dominante.shape[0] - len(fases_validas)}")

Fases totales antes:     19149
Fases válidas (dominante >80%): 18548
Fases descartadas (repartidas): 601


In [14]:
imputaciones_maquina_dominante = fases_validas.shape[0]

## Tabla final imputaciones: una instancia por cada fase
Finalmente voy a construir la tabla, con una fila por fase válida, con su grupo con la máquina dominante, y sus tiempos reales agregados.

Agregación de imputaciones a nivel de fase

Los datos de ejecución procedentes del MES se registran a nivel de imputación, de modo que una misma fase de fabricación puede acumular múltiples imputaciones correspondientes a las distintas sesiones de trabajo, operarios o intervalos temporales en que se ejecutó. Sin embargo, la unidad de análisis de este estudio es la fase, no la imputación individual. Por ello, resulta necesario agregar las imputaciones de cada fase en un único registro que la represente de forma completa.

Esta agregación responde a dos motivos fundamentales. En primer lugar, la variable objetivo del estudio —el ratio entre el tiempo real y el tiempo teórico— exige que ambos tiempos estén definidos a un mismo nivel. El tiempo teórico se establece en el ERP a nivel de fase, por lo que el tiempo real debe consolidarse igualmente a nivel de fase para que la comparación entre ambos sea coherente; de lo contrario, se dispondría de múltiples tiempos reales enfrentados a un único tiempo teórico, imposibilitando el cálculo del ratio.

En segundo lugar, el modelo que se pretende desarrollar opera sobre la unidad de asignación fase-máquina. Cada observación proporcionada al modelo debe corresponder a una decisión de asignación única. Mantener las imputaciones sin agregar provocaría que una misma fase apareciera representada en varias filas, otorgándole un peso artificialmente elevado en el entrenamiento y distorsionando el aprendizaje de los patrones de idoneidad.

La agregación se ha diseñado, además, preservando la información relevante contenida en las imputaciones individuales. Más allá de la suma de los tiempos, se conservan atributos constantes dentro de la fase (como la orden de fabricación o el material) y se derivan variables agregadas de interés analítico, garantizando que la reducción a un único registro por fase no suponga una pérdida de información útil para el modelado.

In [15]:
display(df_imput_productivas.head(0))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid


Tiempo bruto y tiempo efectivo: corrección por dedicación

El MES registra el tiempo de cada imputación como tiempo de reloj (el transcurrido entre inicio y fin), sin considerar la atención real dedicada a la tarea. Como es habitual que un operario gestione varias máquinas a la vez —abre una operación, atiende otra y regresa—, el reloj sigue corriendo aunque no le dedique el 100% de su atención. Para reflejarlo, el MES registra el porcentaje de dedicación de cada imputación, que se reparte entre las tareas concurrentes.

Se definen así dos magnitudes:

- Tiempo bruto (`MinutosReal_total`): el tiempo de reloj sin corregir.      
- Tiempo efectivo (`MinutosReal_total`): el tiempo ponderado por la dedicación (tiempo × dedicación / 100), que representa el trabajo realmente invertido en la fase. 

>La corrección se aplica a nivel de imputación, antes de agregar.

La distinción es clave para la variable objetivo: el tiempo teórico del ERP asume dedicación plena, por lo que solo es comparable con el tiempo efectivo. Usar el bruto penalizaría injustamente a las asignaciones ejecutadas en multitarea, atribuyendo a la máquina un exceso de tiempo debido a una circunstancia organizativa ajena a su idoneidad. Por ello, el ratio se calcula sobre el tiempo efectivo, conservando el bruto solo para trazabilidad.

In [16]:
# 1. me quedo solo con las imputaciones de las fases válidas
imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)].copy()

# 1b. corrijo los tiempos por dedicación (tiempo de reloj → tiempo efectivo)
imput_validas["MinutosReal_efectivo"] = imput_validas["MinutosRealNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100
imput_validas["MinutosPreparacion_efectivo"] = imput_validas["MinutosPreparacionNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100
imput_validas["MinutosProduccion_efectivo"] = imput_validas["MinutosProduccionNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100

# 2. máquina dominante de cada fase (el nodo con más tiempo EFECTIVO)
tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosReal_efectivo"].sum().reset_index()
idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosReal_efectivo"].idxmax()
dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# 3. tiempos agregados por fase
tiempos = imput_validas.groupby("SmFaseUid").agg(
    fecha_fase=("FechaCreacionDte", "min"),   
    smOrdenId=("smOrdenId", "first"),
    smMaterialId=("smMaterialId", "first"),
    smFaseNme=("smFaseNme", "first"),
    smOrdenNme=("smOrdenNme", "first"),
    # tiempos EFECTIVOS (corregidos por dedicación) → los del ratio
    MinutosReal_efectivo=("MinutosReal_efectivo", "sum"),
    MinutosPreparacion_efectivo=("MinutosPreparacion_efectivo", "sum"),
    MinutosProduccion_efectivo=("MinutosProduccion_efectivo", "sum"),
    # tiempos BRUTOS (los conservo para comparar / trazabilidad)
    MinutosReal_total=("MinutosRealNbr", "sum"),
    MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
    MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
    # dedicación como feature
    dedicacion_media=("PorcentajeDedicacionNbr", "mean"),
    n_imputaciones=("MinutosRealNbr", "size"),
).reset_index()

# 4. uno máquina dominante + tiempos en la tabla final (1 fila por fase)
fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

# 5. verificaciones
print(f"Fases en la tabla final: {len(fase_agregada)}")
efectivo_supera = ((fase_agregada["MinutosReal_efectivo"] - fase_agregada["MinutosReal_total"]) > 0.01).sum()
print(f"Fases donde efectivo supera bruto de verdad (>0.01): {efectivo_supera}")  # debe ser 0
display(fase_agregada.sample(5))

Fases en la tabla final: 18548
Fases donde efectivo supera bruto de verdad (>0.01): 0


,SmFaseUid,MaquinaDominante,fecha_fase,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones
3103,2B7FF9EC-6AA4-4931-9EE5-A929ECE4A7F1,TP02,2025-11-18 11:28:10.360,OF2511296,PE1XA3X0364,MTP,Portaelectrodos 103124322,42.866660,42.866660,0.0,42.86666,42.86666,0.0,50.000000,4
15954,DC0A36B1-5790-415B-8903-F88EAA38AAD4,TP02,2025-10-30 07:00:37.603,OF2510828,CODXA2X0020,MTP,Strombrücke A8043035,44.866670,44.866670,0.0,44.86667,44.86667,0.0,50.000000,2
12351,AA0B85C4-8AFE-4007-BD80-894A8D7C460D,CM06,2025-08-11 06:33:12.550,OF2507671,PE0XA2X0223,PE01,Elektrodenaufnahme F704500528853,161.516670,161.516670,0.0,161.51667,161.51667,0.0,50.000000,2
5713,4FB8CE29-4FB5-438E-84AD-1569B24DD70F,TP02,2026-06-29 20:12:34.363,OF2605254,PE2XA3X0306,MTP,COPPER PART Holder ARO4992E9656,23.259651,23.259651,0.0,63.13333,63.13333,0.0,18.421055,2
3483,30B55D9A-4C0A-4118-9791-B803A0BD5E36,TN09,2025-12-12 11:16:17.260,OF2511989,PE1XA3X0124,TPP,Portaelectrodos IM00065214(KA872421S),51.600000,51.600000,0.0,51.60000,51.60000,0.0,50.000000,2


### Revisar el correcto traspaso de tiempos en las agrupaciones:
Los minutos totales se calcula de la siguiente manera:
`MinutosReal_total = MinutosPreparacion_total + MinutosProduccion_total`    
Entonces si calculo la diferencia entre Preparación y Producción debe dar 0


In [17]:
# ============================================================
# DEPURACIÓN: fases con tiempo inflado por MES sin cerrar
# ============================================================
# Criterio: un tiempo de preparación o producción desproporcionado
# (>= 480 min ≈ 1 turno) junto con un nº elevado de imputaciones
# indica que la máquina quedó abierta en el MES acumulando tiempo
# sin trabajo real.

UMBRAL_TIEMPO = 480          # minutos (1 turno de 8h) en prep o prod
UMBRAL_IMPUTACIONES = 3     # nº de imputaciones sospechoso

fase_agregada["mes_sin_cerrar"] = (
    (
        (fase_agregada["MinutosPreparacion_total"] >= UMBRAL_TIEMPO)
        | (fase_agregada["MinutosProduccion_total"] >= UMBRAL_TIEMPO)
    )
    & (fase_agregada["n_imputaciones"] > UMBRAL_IMPUTACIONES)
)

# reporte antes de eliminar
sospechosas = fase_agregada[fase_agregada["mes_sin_cerrar"]]
print(f"Fases marcadas como MES sin cerrar: {len(sospechosas)}")
print(sospechosas.nlargest(10, "MinutosProduccion_total")[
    ["SmFaseUid", "MaquinaDominante", "smFaseNme",
     "MinutosPreparacion_total", "MinutosProduccion_total", "n_imputaciones"]
].to_string(index=False))

antes = len(fase_agregada)
fase_agregada = fase_agregada[~fase_agregada["mes_sin_cerrar"]]
print(f"\nFases antes: {antes} | eliminadas: {antes - len(fase_agregada)} | después: {len(fase_agregada)}")

fase_agregada = fase_agregada.drop(columns=["mes_sin_cerrar"])

Fases marcadas como MES sin cerrar: 1159
                           SmFaseUid MaquinaDominante smFaseNme  MinutosPreparacion_total  MinutosProduccion_total  n_imputaciones
5D715E06-67F3-451C-AE5E-982E0083402F             CM04    CV-ALU                4968.85000              32412.53333              77
EDD89008-F84F-427B-802D-24BFEA25DEE4             CM17     MEC03                 585.76666              30921.84999              74
B1378B1E-8601-4B4F-A08C-501D8A0703E6             CM17     MEC03                 585.29999              30917.48332              74
BDD4BE9B-9BCF-4BCF-BCF4-E1BBB9E780DC             CM17     MEC03                 585.71666              30917.48332              74
1A5CBCD3-3A66-4C09-82A2-ADF8A9BE0E3A             CM17     MEC03                2243.95000              28929.40000              74
3A56FC28-C0C7-4219-9701-C91B548BB507             CM17     MEC03                2243.95000              28929.40000              74
68416ED9-521E-405A-8DB1-0D42D2CDDF84      

In [18]:
df_fases_sin_mal_cerrar = fase_agregada.shape[0]

In [19]:
display(fase_agregada.sample(3))

,SmFaseUid,MaquinaDominante,fecha_fase,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones
14848,CC941221-A1E5-4C8A-8EB5-0923A24D40AA,TP02,2026-02-02 16:53:08.073,OF2600698,PE2XNCX0077,MTP,Allonge F704500091688,69.576926,69.576926,0.0,90.45000,90.45000,0.0,38.46154,2
6681,5CA47651-C430-48DD-BCCA-1429CDBBFE6D,CM12,2026-01-19 13:21:54.217,OF2500456,CFFXA1X0009,MEC03,Shunt IWS000984,2.233330,2.233330,0.0,2.23333,2.23333,0.0,50.00000,2
5943,52EABE68-AD26-4650-92C2-E5D77D630C98,CM05,2025-12-12 06:01:10.340,OF2511774,PE2XNCX0096,PE02,ALLONGE COUDEE 103134365,91.216670,91.216670,0.0,91.21667,91.21667,0.0,50.00000,2


> Hay imputaciones con un ruido inmanejable, voy a tener que eliminarlas.

Durante la validación de los tiempos registrados en el sistema de captura de planta (MES) se observó, en un subconjunto de fases, una incoherencia entre el tiempo real imputado y la suma de sus componentes de preparación y producción. Al analizar estos casos, se identificó que las desviaciones más severas se concentraban en fases con un número anormalmente elevado de imputaciones. La inspección manual de estas fases reveló el origen del problema: en determinadas ocasiones, una máquina permanecía con la orden abierta en el MES tras finalizar el trabajo efectivo, de modo que el sistema continuaba acumulando imputaciones de tiempo real de forma automática, sin actividad productiva real asociada. Este tiempo espurio inflaba de forma muy acusada el tiempo real de la fase.

Dado que el tiempo real constituye el numerador de la variable objetivo del estudio (el ratio entre tiempo real y teórico), estos registros habrían distorsionado gravemente el cálculo de la idoneidad, atribuyendo desviaciones elevadas a asignaciones que en realidad fueron correctas y cuyo exceso de tiempo se debía únicamente a un fallo en el cierre del registro. Por este motivo, y tras verificar individualmente su naturaleza, se han descartado las fases afectadas por este fenómeno. El criterio de identificación combinó la magnitud de la desviación con el número de imputaciones de la fase, lo que permitió aislar los registros defectuosos sin eliminar aquellas fases cuya ligera desviación responde a las condiciones normales de imputación del trabajo. Tras esta depuración, el conjunto de análisis queda constituido por 17.389 fases con tiempos fiables.

In [20]:
# Voy añadir el grupo de la máquina a la tabla final de imputaciones agregadas por fase
df_fases_MES = fase_agregada.merge(maquina_grupo, left_on="MaquinaDominante", right_on="CodigoMaquina", how="left")

In [21]:
# Verificar si alguna fase se quedo sin grupo máquina:
print(f"Fases sin grupo tras el merge: {df_fases_MES['NombreGrupo'].isna().sum()}")

Fases sin grupo tras el merge: 0


In [22]:
df_fases_MES = df_fases_MES.drop("CodigoMaquina", axis=1)

display(df_fases_MES.sample(1))
display(f"Grupos de máquinas: {len(df_fases_MES["CodigoGrupo"].unique())}")

,SmFaseUid,MaquinaDominante,fecha_fase,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
11134,A3ADAD42-3172-48A2-A03C-489E0A545F77,TP02,2025-10-13 13:35:34.463,OF2510127,PE2XA2X0134,MTP,ALLONGE COUDÉE 262045,37.01667,37.01666,0.0,37.01667,37.01666,0.0,66.666667,3,a045db52-b7d0-4011-b402-e7f63d5e4d3a,MTP IMSA MF800C-HD Màquina Taladres Profunds,7d29197b-e1a1-4c19-bf95-7fa3b96ed500,MTP,Màquines de Taladrat Profund


'Grupos de máquinas: 9'

In [23]:
df_fases_MES.to_parquet("res//procesed/df_fases_MES.parquet")

## Evolución de las imputaciones:

In [24]:
# importo variables locales
with open("res/procesed/variables/1.Tratamiento_Imputaciones.pkl", "rb") as f:
    variables = pickle.load(f)
df_inicial = variables["Imputaciones iniciales"]
df_intervalo_timepos = variables["Imputaciones dentro del intervalo temporal"]



In [25]:
variables = {
    "Imputaciones en máquinas productivas": df_imputations_productivas,
    "Fases en máquinas productivas": total_fases_productivas,
    "Fases válidas (dominante >80%)": imputaciones_maquina_dominante,
    "Fases sin imputaciones mal cerradas": df_fases_sin_mal_cerrar
   
}
with open("res/procesed/variables/3.Tratamiento_Imputaciones.pkl", "wb") as f:
    pickle.dump(variables, f) 

In [26]:
tabla_resultados = pd.DataFrame({
    "Imputaciones iniciales": [df_inicial],
    "Imputaciones dentro del intervalo temporal": [df_intervalo_timepos],
    "Imputaciones en máquinas productivas": [df_imputations_productivas], # nueva variable
}).T

tabla_resultados = tabla_resultados.rename(columns={0: "Número de imputaciones"})

print("=======================================================")
print("Resultados del notebook:")
print("=======================================================")
display(tabla_resultados)
print("=======================================================")


tabla_resultados_fases = pd.DataFrame({
    "Fases en máquinas productivas": [total_fases_productivas], # nueva variable
    "Fases válidas (dominante >80%)": [imputaciones_maquina_dominante], # nueva variable
    "Fases sin imputaciones mal cerradas": [df_fases_sin_mal_cerrar] # nueva variable
}).T
tabla_resultados_fases = tabla_resultados_fases.rename(columns={0: "Número de fases"})

print("=======================================================")
print("Resultados del notebook:")
print("=======================================================")
display(tabla_resultados_fases)
print("=======================================================")



Resultados del notebook:


,Número de imputaciones
Imputaciones iniciales,196576
Imputaciones dentro del intervalo temporal,140485
Imputaciones en máquinas productivas,59570


Resultados del notebook:


,Número de fases
Fases en máquinas productivas,19149
Fases válidas (dominante >80%),18548
Fases sin imputaciones mal cerradas,17389
